<a href="https://colab.research.google.com/github/Misha-private/Demo-repo/blob/main/ML_3DVAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
def is_drive_mounted():
    return os.path.exists('/content/drive')
if not is_drive_mounted(): drive.mount('/content/drive')

Mounted at /content/drive


# Build ML3DVAR dataset

In [ ]:
# ============================================================
# build_ml3dvar_dataset_1layer.py
# ------------------------------------------------------------
# Build ML-3DVAR / U-Net DA dataset from FD shallow-water data.
#
# Input channels:
#   [eta_b, u_b, v_b, eta_obs, u_obs, v_obs, mask]
#
# Target:
#   [eta_true, u_true, v_true]
#
# Output:
#   /content/drive/MyDrive/AI_4DVAR2/ml3dvar_dataset_1L
# ============================================================

import os
import glob
import csv
import numpy as np

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT_IN  = "/content/drive/MyDrive/AI_4DVAR"
ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

DATA_DIR_CANDIDATES = [
    f"{ROOT_IN}/klein_ckpt_1L",
    f"{ROOT_IN}/klein_ckpt_AL_centers_colloc",
    f"{ROOT_IN}/klein_ckpt_AL_centers",
    f"{ROOT_IN}/klein_ckpt_1L_colloc",
]

OUT_DIR = f"{ROOT_OUT}/ml3dvar_dataset_1L"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Dataset controls
# ------------------------------------------------------------
OBS_FRAC = 0.03

PERT_ETA_STD = 2.0
PERT_UV_STD  = 0.2

N_AUG_PER_SNAPSHOT = 4

TRAIN_FRAC = 0.75
VAL_FRAC   = 0.15
TEST_FRAC  = 0.10

SEED = 42
rng = np.random.default_rng(SEED)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_snapshots(data_dir):
    if not os.path.exists(data_dir):
        return 0
    files = glob.glob(os.path.join(data_dir, "*", "klein_step_*.npz"))
    return len(files)

def autodetect_data_dir(candidates):
    best_dir = None
    best_count = -1

    print("[AutoDetect] checking data roots...")
    for d in candidates:
        n = count_snapshots(d)
        print(f"  {d}: {n} snapshots")
        if n > best_count:
            best_count = n
            best_dir = d

    if best_dir is None or best_count <= 0:
        raise RuntimeError("No valid FD snapshot directory found.")

    print("[AutoDetect] using:", best_dir)
    return best_dir

def make_obs_mask(ny, nx, obs_frac, rng):
    mask = rng.random((ny, nx)) < obs_frac
    return mask.astype(np.float32)

def make_background(truth, rng):
    """
    truth: [3, NY, NX]
    """
    bg = truth.copy()

    bg[0] += rng.normal(0.0, PERT_ETA_STD, size=truth[0].shape).astype(np.float32)
    bg[1] += rng.normal(0.0, PERT_UV_STD,  size=truth[1].shape).astype(np.float32)
    bg[2] += rng.normal(0.0, PERT_UV_STD,  size=truth[2].shape).astype(np.float32)

    return bg.astype(np.float32)

def build_sample(truth, rng):
    """
    truth: [3, NY, NX]
    returns:
      x: [7, NY, NX]
      y: [3, NY, NX]
    """
    ny, nx = truth.shape[-2], truth.shape[-1]

    bg = make_background(truth, rng)

    mask = make_obs_mask(ny, nx, OBS_FRAC, rng)

    obs = np.zeros_like(truth, dtype=np.float32)
    obs[0] = truth[0] * mask
    obs[1] = truth[1] * mask
    obs[2] = truth[2] * mask

    x = np.concatenate(
        [
            bg,
            obs,
            mask[None, :, :],
        ],
        axis=0,
    ).astype(np.float32)

    y = truth.astype(np.float32)

    return x, y, bg, obs, mask

def parse_ic_key(path):
    return os.path.basename(os.path.dirname(path))

def parse_step(path):
    base = os.path.basename(path)
    return base.replace("klein_step_", "").replace(".npz", "")

# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
DATA_DIR = autodetect_data_dir(DATA_DIR_CANDIDATES)

all_files = sorted(glob.glob(os.path.join(DATA_DIR, "*", "klein_step_*.npz")))
print(f"[Dataset] found {len(all_files)} snapshots")

# Shuffle by snapshot before splitting
rng.shuffle(all_files)

n_total_snapshots = len(all_files)
n_train = int(TRAIN_FRAC * n_total_snapshots)
n_val   = int(VAL_FRAC * n_total_snapshots)

split_files = {
    "train": all_files[:n_train],
    "val":   all_files[n_train:n_train+n_val],
    "test":  all_files[n_train+n_val:],
}

print("[Split]")
for split, files in split_files.items():
    print(f"  {split}: {len(files)} snapshots")

manifest_rows = []

sample_counter = 0

for split, files in split_files.items():
    split_dir = os.path.join(OUT_DIR, split)
    os.makedirs(split_dir, exist_ok=True)

    print(f"\n[Build] split={split}")

    for i, fpath in enumerate(files):
        z = np.load(fpath)

        truth = np.stack(
            [
                z["eta"],
                z["uc"],
                z["vc"],
            ],
            axis=0,
        ).astype(np.float32)

        ic_key = parse_ic_key(fpath)
        step_key = parse_step(fpath)

        for aug in range(N_AUG_PER_SNAPSHOT):
            x, y, bg, obs, mask = build_sample(truth, rng)

            sample_name = f"{split}_{sample_counter:06d}_{ic_key}_step{step_key}_aug{aug}.npz"
            out_path = os.path.join(split_dir, sample_name)

            np.savez_compressed(
                out_path,
                x=x,
                y=y,
                bg=bg,
                obs=obs,
                mask=mask,
                ic_key=ic_key,
                step_key=step_key,
                source_file=fpath,
                obs_frac=np.float32(OBS_FRAC),
                pert_eta_std=np.float32(PERT_ETA_STD),
                pert_uv_std=np.float32(PERT_UV_STD),
            )

            manifest_rows.append(
                {
                    "sample_id": sample_counter,
                    "split": split,
                    "ic_key": ic_key,
                    "step_key": step_key,
                    "aug": aug,
                    "path": out_path,
                    "source_file": fpath,
                    "obs_frac": OBS_FRAC,
                    "pert_eta_std": PERT_ETA_STD,
                    "pert_uv_std": PERT_UV_STD,
                }
            )

            sample_counter += 1

        if (i + 1) % 20 == 0:
            print(f"  processed {i+1}/{len(files)} snapshots")

# ------------------------------------------------------------
# Save manifest
# ------------------------------------------------------------
manifest_csv = os.path.join(OUT_DIR, "manifest.csv")

with open(manifest_csv, "w", newline="") as f:
    keys = list(manifest_rows[0].keys())
    w = csv.DictWriter(f, fieldnames=keys)
    w.writeheader()
    for row in manifest_rows:
        w.writerow(row)

# ------------------------------------------------------------
# Save small metadata file
# ------------------------------------------------------------
meta_path = os.path.join(OUT_DIR, "metadata.txt")

with open(meta_path, "w") as f:
    f.write("ML-3DVAR 1-layer SWE dataset\n")
    f.write(f"DATA_DIR = {DATA_DIR}\n")
    f.write(f"OBS_FRAC = {OBS_FRAC}\n")
    f.write(f"PERT_ETA_STD = {PERT_ETA_STD}\n")
    f.write(f"PERT_UV_STD = {PERT_UV_STD}\n")
    f.write(f"N_AUG_PER_SNAPSHOT = {N_AUG_PER_SNAPSHOT}\n")
    f.write(f"TRAIN_FRAC = {TRAIN_FRAC}\n")
    f.write(f"VAL_FRAC = {VAL_FRAC}\n")
    f.write(f"TEST_FRAC = {TEST_FRAC}\n")
    f.write(f"total_samples = {sample_counter}\n")

print("\n[Done]")
print("Output directory:", OUT_DIR)
print("Manifest:", manifest_csv)
print("Total samples:", sample_counter)

[AutoDetect] checking data roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L: 196 snapshots
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc: 0 snapshots
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers: 0 snapshots
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc: 0 snapshots
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[Dataset] found 196 snapshots
[Split]
  train: 147 snapshots
  val: 29 snapshots
  test: 20 snapshots

[Build] split=train
  processed 20/147 snapshots
  processed 40/147 snapshots
  processed 60/147 snapshots
  processed 80/147 snapshots
  processed 100/147 snapshots
  processed 120/147 snapshots
  processed 140/147 snapshots

[Build] split=val
  processed 20/29 snapshots

[Build] split=test
  processed 20/20 snapshots

[Done]
Output directory: /content/drive/MyDrive/AI_4DVAR2/ml3dvar_dataset_1L
Manifest: /content/drive/MyDrive/AI_4DVAR2/ml3dvar_dataset_1L/manifest.csv
Total samples: 784


# U-Net ML-3DVAR

In [ ]:
# ============================================================
# train_ml3dvar_unet_1layer.py
# ------------------------------------------------------------
# Train U-Net style ML-3DVAR analysis model for 1-layer SWE.
#
# Dataset input x:
#   [eta_b, u_b, v_b, eta_obs, u_obs, v_obs, mask]
#
# Target y:
#   [eta_true, u_true, v_true]
#
# Network predicts increment:
#   dx = UNet(x)
#   analysis = background + dx
#
# Output:
#   /content/drive/MyDrive/AI_4DVAR2/ml3dvar_unet_1L_ckpt
# ============================================================

import os
import glob
import csv
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

DATASET_DIR = f"{ROOT_OUT}/ml3dvar_dataset_1L"
CKPT_DIR = f"{ROOT_OUT}/ml3dvar_unet_1L_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Hyperparameters
# ------------------------------------------------------------
BATCH_SIZE = 4
EPOCHS = 80
LR = 2e-4
WEIGHT_DECAY = 1e-6

BASE_CH = 48

LAMBDA_SMOOTH = 1e-4
LAMBDA_UV_INC = 1e-2
LAMBDA_INC = 1e-4

GRAD_CLIP = 1.0
PRINT_EVERY = 20

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class ML3DVARDataset(Dataset):
    def __init__(self, root, split):
        self.files = sorted(glob.glob(os.path.join(root, split, "*.npz")))
        if len(self.files) == 0:
            raise RuntimeError(f"No files found for split={split} in {root}")
        print(f"[Dataset] {split}: {len(self.files)} samples")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        z = np.load(self.files[idx])

        x = z["x"].astype(np.float32)  # [7, NY, NX]
        y = z["y"].astype(np.float32)  # [3, NY, NX]

        bg = x[:3].astype(np.float32)

        return {
            "x": torch.from_numpy(x),
            "y": torch.from_numpy(y),
            "bg": torch.from_numpy(bg),
        }

train_ds = ML3DVARDataset(DATASET_DIR, "train")
val_ds   = ML3DVARDataset(DATASET_DIR, "val")

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8 if out_ch >= 8 else 1, out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8 if out_ch >= 8 else 1, out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)

class UNetIncrement(nn.Module):
    def __init__(self, in_ch=7, out_ch=3, base_ch=48):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base_ch)          # 48
        self.enc2 = ConvBlock(base_ch, base_ch * 2)    # 96
        self.enc3 = ConvBlock(base_ch * 2, base_ch * 4) # 192
        self.enc4 = ConvBlock(base_ch * 4, base_ch * 8) # 384

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base_ch * 8, base_ch * 8)

        self.up4 = nn.ConvTranspose2d(base_ch * 8, base_ch * 8, 2, stride=2)
        self.dec4 = ConvBlock(base_ch * 16, base_ch * 4)

        self.up3 = nn.ConvTranspose2d(base_ch * 4, base_ch * 4, 2, stride=2)
        self.dec3 = ConvBlock(base_ch * 8, base_ch * 2)

        self.up2 = nn.ConvTranspose2d(base_ch * 2, base_ch * 2, 2, stride=2)
        self.dec2 = ConvBlock(base_ch * 4, base_ch)

        self.up1 = nn.ConvTranspose2d(base_ch, base_ch, 2, stride=2)
        self.dec1 = ConvBlock(base_ch * 2, base_ch)

        self.out = nn.Conv2d(base_ch, out_ch, 1)

        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        e1 = self.enc1(x)              # [B, 48, 128, 256]
        e2 = self.enc2(self.pool(e1))  # [B, 96, 64, 128]
        e3 = self.enc3(self.pool(e2))  # [B, 192, 32, 64]
        e4 = self.enc4(self.pool(e3))  # [B, 384, 16, 32]

        b = self.bottleneck(self.pool(e4))  # [B, 384, 8, 16]

        d4 = self.up4(b)                    # [B, 384, 16, 32]
        d4 = torch.cat([d4, e4], dim=1)     # [B, 768, 16, 32]
        d4 = self.dec4(d4)                  # [B, 192, 16, 32]

        d3 = self.up3(d4)                   # [B, 192, 32, 64]
        d3 = torch.cat([d3, e3], dim=1)     # [B, 384, 32, 64]
        d3 = self.dec3(d3)                  # [B, 96, 32, 64]

        d2 = self.up2(d3)                   # [B, 96, 64, 128]
        d2 = torch.cat([d2, e2], dim=1)     # [B, 192, 64, 128]
        d2 = self.dec2(d2)                  # [B, 48, 64, 128]

        d1 = self.up1(d2)                   # [B, 48, 128, 256]
        d1 = torch.cat([d1, e1], dim=1)     # [B, 96, 128, 256]
        d1 = self.dec1(d1)                  # [B, 48, 128, 256]

        dx = self.out(d1)
        return dx

model = UNetIncrement(in_ch=7, out_ch=3, base_ch=BASE_CH).to(device)

print(model)

# ------------------------------------------------------------
# Loss helpers
# ------------------------------------------------------------
def smoothness_loss(dx):
    ddx = dx[:, :, :, 1:] - dx[:, :, :, :-1]
    ddy = dx[:, :, 1:, :] - dx[:, :, :-1, :]
    return (ddx ** 2).mean() + (ddy ** 2).mean()

def rmse_torch(a, b):
    return torch.sqrt(torch.mean((a - b) ** 2))

def compute_losses(batch, model):
    x = batch["x"].to(device, non_blocking=True)
    y = batch["y"].to(device, non_blocking=True)
    bg = batch["bg"].to(device, non_blocking=True)

    dx = model(x)
    ana = bg + dx

    loss_state = ((ana - y) ** 2).mean()
    loss_smooth = smoothness_loss(dx)
    loss_uv_inc = (dx[:, 1:] ** 2).mean()
    loss_inc = (dx ** 2).mean()

    loss = (
        loss_state
        + LAMBDA_SMOOTH * loss_smooth
        + LAMBDA_UV_INC * loss_uv_inc
        + LAMBDA_INC * loss_inc
    )

    with torch.no_grad():
        bg_rmse = rmse_torch(bg, y)
        ana_rmse = rmse_torch(ana, y)
        eta_rmse = rmse_torch(ana[:, 0], y[:, 0])
        u_rmse = rmse_torch(ana[:, 1], y[:, 1])
        v_rmse = rmse_torch(ana[:, 2], y[:, 2])

    return loss, {
        "loss_state": loss_state,
        "loss_smooth": loss_smooth,
        "loss_uv_inc": loss_uv_inc,
        "loss_inc": loss_inc,
        "bg_rmse": bg_rmse,
        "ana_rmse": ana_rmse,
        "eta_rmse": eta_rmse,
        "u_rmse": u_rmse,
        "v_rmse": v_rmse,
    }

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=8,
)

# ------------------------------------------------------------
# Train / validate
# ------------------------------------------------------------
history = []
best_val = float("inf")

def run_epoch(split, loader, train_mode):
    if train_mode:
        model.train()
    else:
        model.eval()

    sums = {
        "total": 0.0,
        "state": 0.0,
        "smooth": 0.0,
        "uv_inc": 0.0,
        "inc": 0.0,
        "bg_rmse": 0.0,
        "ana_rmse": 0.0,
        "eta_rmse": 0.0,
        "u_rmse": 0.0,
        "v_rmse": 0.0,
    }

    n = 0

    for ib, batch in enumerate(loader):
        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        if train_mode:
            loss, parts = compute_losses(batch, model)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        else:
            with torch.no_grad():
                loss, parts = compute_losses(batch, model)

        bs = batch["x"].shape[0]
        n += bs

        sums["total"] += float(loss.detach().cpu()) * bs
        sums["state"] += float(parts["loss_state"].detach().cpu()) * bs
        sums["smooth"] += float(parts["loss_smooth"].detach().cpu()) * bs
        sums["uv_inc"] += float(parts["loss_uv_inc"].detach().cpu()) * bs
        sums["inc"] += float(parts["loss_inc"].detach().cpu()) * bs
        sums["bg_rmse"] += float(parts["bg_rmse"].detach().cpu()) * bs
        sums["ana_rmse"] += float(parts["ana_rmse"].detach().cpu()) * bs
        sums["eta_rmse"] += float(parts["eta_rmse"].detach().cpu()) * bs
        sums["u_rmse"] += float(parts["u_rmse"].detach().cpu()) * bs
        sums["v_rmse"] += float(parts["v_rmse"].detach().cpu()) * bs

        if train_mode and ib % PRINT_EVERY == 0:
            print(
                f"[{split} batch {ib:04d}/{len(loader):04d}] "
                f"loss={loss.item():.6e} "
                f"state={parts['loss_state'].item():.6e} "
                f"bg_rmse={parts['bg_rmse'].item():.4f} "
                f"ana_rmse={parts['ana_rmse'].item():.4f}"
            )

    for k in sums:
        sums[k] /= max(n, 1)

    return sums

for epoch in range(EPOCHS):
    t0 = time.time()

    train_stats = run_epoch("train", train_loader, train_mode=True)
    val_stats = run_epoch("val", val_loader, train_mode=False)

    scheduler.step(val_stats["total"])

    lr_now = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "lr": lr_now,
        "train_total": train_stats["total"],
        "train_state": train_stats["state"],
        "train_smooth": train_stats["smooth"],
        "train_uv_inc": train_stats["uv_inc"],
        "train_inc": train_stats["inc"],
        "train_bg_rmse": train_stats["bg_rmse"],
        "train_ana_rmse": train_stats["ana_rmse"],
        "val_total": val_stats["total"],
        "val_state": val_stats["state"],
        "val_smooth": val_stats["smooth"],
        "val_uv_inc": val_stats["uv_inc"],
        "val_inc": val_stats["inc"],
        "val_bg_rmse": val_stats["bg_rmse"],
        "val_ana_rmse": val_stats["ana_rmse"],
        "val_eta_rmse": val_stats["eta_rmse"],
        "val_u_rmse": val_stats["u_rmse"],
        "val_v_rmse": val_stats["v_rmse"],
    }

    history.append(row)

    print(
        f"\nEpoch {epoch:03d} | "
        f"train={train_stats['total']:.6e} | "
        f"val={val_stats['total']:.6e} | "
        f"val bg={val_stats['bg_rmse']:.4f} | "
        f"val ana={val_stats['ana_rmse']:.4f} | "
        f"lr={lr_now:.2e} | "
        f"time={time.time() - t0:.1f}s\n"
    )

    if val_stats["total"] < best_val:
        best_val = val_stats["total"]
        ckpt_path = os.path.join(CKPT_DIR, "best_model.pt")
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": {
                    "BASE_CH": BASE_CH,
                    "LR": LR,
                    "BATCH_SIZE": BATCH_SIZE,
                    "LAMBDA_SMOOTH": LAMBDA_SMOOTH,
                    "LAMBDA_UV_INC": LAMBDA_UV_INC,
                    "LAMBDA_INC": LAMBDA_INC,
                    "DATASET_DIR": DATASET_DIR,
                },
                "best_val": best_val,
            },
            ckpt_path,
        )
        print("[Save] best:", ckpt_path)

    # save history every epoch
    hist_csv = os.path.join(CKPT_DIR, "loss_history.csv")
    with open(hist_csv, "w", newline="") as f:
        keys = list(history[0].keys())
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in history:
            w.writerow(r)

# final checkpoint
final_ckpt = os.path.join(CKPT_DIR, "final_model.pt")
torch.save(
    {
        "epoch": EPOCHS - 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": {
            "BASE_CH": BASE_CH,
            "LR": LR,
            "BATCH_SIZE": BATCH_SIZE,
            "LAMBDA_SMOOTH": LAMBDA_SMOOTH,
            "LAMBDA_UV_INC": LAMBDA_UV_INC,
            "LAMBDA_INC": LAMBDA_INC,
            "DATASET_DIR": DATASET_DIR,
        },
        "best_val": best_val,
    },
    final_ckpt,
)

print("[Done] final:", final_ckpt)
print("[Done] best val:", best_val)

# ------------------------------------------------------------
# Plot loss curves
# ------------------------------------------------------------
epochs = np.array([r["epoch"] for r in history])
train_total = np.array([r["train_total"] for r in history])
val_total = np.array([r["val_total"] for r in history])
train_ana = np.array([r["train_ana_rmse"] for r in history])
val_ana = np.array([r["val_ana_rmse"] for r in history])
val_bg = np.array([r["val_bg_rmse"] for r in history])

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_total, label="train total")
plt.plot(epochs, val_total, label="val total")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ML-3DVAR U-Net training loss")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "loss_curve.png"), dpi=160)
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(epochs, val_bg, label="val background RMSE")
plt.plot(epochs, val_ana, label="val analysis RMSE")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("Validation RMSE: background vs ML-3DVAR analysis")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "val_rmse_curve.png"), dpi=160)
plt.close()

print("[Done] plots saved in:", CKPT_DIR)

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[Dataset] train: 588 samples
[Dataset] val: 116 samples
UNetIncrement(
  (enc1): ConvBlock(
    (net): Sequential(
      (0): Conv2d(7, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): GroupNorm(8, 48, eps=1e-05, affine=True)
      (2): GELU(approximate='none')
      (3): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): GroupNorm(8, 48, eps=1e-05, affine=True)
      (5): GELU(approximate='none')
    )
  )
  (enc2): ConvBlock(
    (net): Sequential(
      (0): Conv2d(48, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): GroupNorm(8, 96, eps=1e-05, affine=True)
      (2): GELU(approximate='none')
      (3): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): GroupNorm(8, 96, eps=1e-05, affine=True)
      (5): GELU(approximate='none')
    )
  )
  (enc3): ConvBlock(
    (net): Sequential(
      (0): Conv2d(96, 192, kernel_size=(3, 3), st

In [ ]:
# Evaluation

In [ ]:
# ============================================================
# eval_ml3dvar_vs_ml4dvar_1layer.py
# ============================================================

# Compares:
#   1. Background rollout
#   2. ML-3DVAR U-Net analysis rollout
#   3. ML-4DVAR optimized analysis rollout
#
# Uses final emulator:
#   rescnn_b12_c96_lr1e4_t6_p4_keweak
#
# Outputs:
#   /content/drive/MyDrive/AI_4DVAR2/eval_ml3dvar_vs_ml4dvar
# ============================================================

import os, glob, csv, time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT_IN  = "/content/drive/MyDrive/AI_4DVAR"
ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

DATA_DIR = f"{ROOT_IN}/klein_ckpt_1L"

EMULATOR_CKPT = (
    f"{ROOT_OUT}/training_runs/"
    f"rescnn_b12_c96_lr1e4_t6_p4_keweak/final_model.pt"
)

ML3DVAR_CKPT = (
    f"{ROOT_OUT}/ml3dvar_unet_1L_ckpt/best_model.pt"
)

OUT_DIR = f"{ROOT_OUT}/eval_ml3dvar_vs_ml4dvar"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Constants / controls
# ------------------------------------------------------------
NX = 256
NY = 128
DT_MACRO = 200.0 * 30.0

CASE_LIST = [
    "gauss_00",
    "test_modon_00",
    "test_rh_00",
    "wave_00",
]

START_INDEX = 0
WINDOW_STEPS = 4

OBS_FRAC = 0.03
PERT_ETA_STD = 2.0
PERT_UV_STD  = 0.2

# ML-4DVAR settings
N_OPT = 250
LR_4DVAR = 2e-2
LAMBDA_BG = 0.10
LAMBDA_SMOOTH = 1e-3
LAMBDA_UV_INC = 0.50
PRINT_EVERY = 50

# ------------------------------------------------------------
# Emulator model
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class EmulatorResCNN(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=12):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        dilations = [1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2]
        self.resnet = nn.Sequential(
            *[ResBlock(feat_ch, dilation=dilations[i % len(dilations)])
              for i in range(n_blocks)]
        )

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        # Present in checkpoint; not used here.
        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        return self.resnet(self.stem(x))

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def forward_step(self, x):
        feat = self.encode(x)
        xdot = self.grid_tendency(feat)
        return x + DT_MACRO * xdot

# ------------------------------------------------------------
# ML-3DVAR U-Net model
# ------------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        groups = 8 if out_ch >= 8 else 1
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(groups, out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(groups, out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)

class UNetIncrement(nn.Module):
    def __init__(self, in_ch=7, out_ch=3, base_ch=48):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base_ch)
        self.enc2 = ConvBlock(base_ch, base_ch * 2)
        self.enc3 = ConvBlock(base_ch * 2, base_ch * 4)
        self.enc4 = ConvBlock(base_ch * 4, base_ch * 8)

        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base_ch * 8, base_ch * 8)

        self.up4 = nn.ConvTranspose2d(base_ch * 8, base_ch * 8, 2, stride=2)
        self.dec4 = ConvBlock(base_ch * 16, base_ch * 4)

        self.up3 = nn.ConvTranspose2d(base_ch * 4, base_ch * 4, 2, stride=2)
        self.dec3 = ConvBlock(base_ch * 8, base_ch * 2)

        self.up2 = nn.ConvTranspose2d(base_ch * 2, base_ch * 2, 2, stride=2)
        self.dec2 = ConvBlock(base_ch * 4, base_ch)

        self.up1 = nn.ConvTranspose2d(base_ch, base_ch, 2, stride=2)
        self.dec1 = ConvBlock(base_ch * 2, base_ch)

        self.out = nn.Conv2d(base_ch, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def load_sequence(ic_key):
    ic_dir = os.path.join(DATA_DIR, ic_key)
    files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))

    if len(files) < START_INDEX + WINDOW_STEPS + 1:
        raise RuntimeError(f"Not enough snapshots for {ic_key}")

    seq = []
    for f in files[START_INDEX:START_INDEX + WINDOW_STEPS + 1]:
        z = np.load(f)
        seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))

    return np.stack(seq, axis=0)

def make_obs_mask(ny, nx, obs_frac, seed=7):
    rng = np.random.default_rng(seed)
    return (rng.random((ny, nx)) < obs_frac).astype(np.float32)

def make_background(truth0, seed=42):
    rng = np.random.default_rng(seed)
    bg = truth0.copy()
    bg[0] += rng.normal(0.0, PERT_ETA_STD, size=(NY, NX)).astype(np.float32)
    bg[1] += rng.normal(0.0, PERT_UV_STD,  size=(NY, NX)).astype(np.float32)
    bg[2] += rng.normal(0.0, PERT_UV_STD,  size=(NY, NX)).astype(np.float32)
    return bg.astype(np.float32)

def make_ml3dvar_input(bg_np, truth0_np, mask_np):
    obs = np.zeros_like(truth0_np, dtype=np.float32)
    obs[0] = truth0_np[0] * mask_np
    obs[1] = truth0_np[1] * mask_np
    obs[2] = truth0_np[2] * mask_np

    x = np.concatenate(
        [bg_np, obs, mask_np[None, :, :]],
        axis=0
    ).astype(np.float32)

    return x

def rollout_model(model, x0, steps):
    xs = [x0]
    x = x0
    for _ in range(steps):
        x = model.forward_step(x)
        xs.append(x)
    return xs

def rmse_np(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def total_rmse(e, u, v):
    return float(np.sqrt(e*e + u*u + v*v))

def smoothness_loss(dx):
    ddx = dx[:, :, :, 1:] - dx[:, :, :, :-1]
    ddy = dx[:, :, 1:, :] - dx[:, :, :-1, :]
    return (ddx ** 2).mean() + (ddy ** 2).mean()

def pct_improve(bg, ana):
    return 100.0 * (bg - ana) / max(abs(bg), 1e-12)

# ------------------------------------------------------------
# Load models
# ------------------------------------------------------------
emulator = EmulatorResCNN(feat_ch=96, hidden=192, n_blocks=12).to(device)
ckpt_e = torch.load(EMULATOR_CKPT, map_location=device)
emulator.load_state_dict(ckpt_e["model_state_dict"])
emulator.eval()
for p in emulator.parameters():
    p.requires_grad_(False)

ckpt_3d = torch.load(ML3DVAR_CKPT, map_location=device)
base_ch = ckpt_3d.get("config", {}).get("BASE_CH", 48)

ml3dvar = UNetIncrement(in_ch=7, out_ch=3, base_ch=base_ch).to(device)
ml3dvar.load_state_dict(ckpt_3d["model_state_dict"])
ml3dvar.eval()
for p in ml3dvar.parameters():
    p.requires_grad_(False)

print("[Load] emulator:", EMULATOR_CKPT)
print("[Load] ML-3DVAR:", ML3DVAR_CKPT)
print("[Load] ML-3DVAR base_ch:", base_ch)

# ------------------------------------------------------------
# ML-4DVAR optimizer for one case
# ------------------------------------------------------------
def run_ml4dvar(x_bg, truth, obs_mask):
    dx0 = torch.zeros_like(x_bg, requires_grad=True)
    opt = torch.optim.Adam([dx0], lr=LR_4DVAR)

    loss_hist = []

    for it in range(N_OPT):
        opt.zero_grad(set_to_none=True)

        x0 = x_bg + dx0
        pred_seq = rollout_model(emulator, x0, WINDOW_STEPS)

        loss_obs = torch.tensor(0.0, device=device)
        for k in range(1, WINDOW_STEPS + 1):
            diff = (pred_seq[k] - truth[k:k+1]) * obs_mask
            loss_obs = loss_obs + (diff ** 2).mean()

        loss_bg = (dx0 ** 2).mean()
        loss_sm = smoothness_loss(dx0)
        loss_uv = (dx0[:, 1:] ** 2).mean()

        loss = (
            loss_obs
            + LAMBDA_BG * loss_bg
            + LAMBDA_SMOOTH * loss_sm
            + LAMBDA_UV_INC * loss_uv
        )

        loss.backward()
        opt.step()

        if it % PRINT_EVERY == 0 or it == N_OPT - 1:
            print(
                f"    [4DVAR {it:04d}] "
                f"loss={loss.item():.6e} "
                f"obs={loss_obs.item():.6e} "
                f"bg={loss_bg.item():.6e} "
                f"uv={loss_uv.item():.6e}"
            )

        loss_hist.append([
            it,
            float(loss.detach().cpu()),
            float(loss_obs.detach().cpu()),
            float(loss_bg.detach().cpu()),
            float(loss_sm.detach().cpu()),
            float(loss_uv.detach().cpu()),
        ])

    return x_bg + dx0.detach(), loss_hist

# ------------------------------------------------------------
# Evaluate one rollout
# ------------------------------------------------------------
def rollout_to_numpy(x0_torch):
    with torch.no_grad():
        seq = rollout_model(emulator, x0_torch, WINDOW_STEPS)
    return torch.cat(seq, dim=0).detach().cpu().numpy()

def add_rows(rows, case, method, pred_np, truth_np):
    for k in range(WINDOW_STEPS + 1):
        e = rmse_np(pred_np[k, 0], truth_np[k, 0])
        u = rmse_np(pred_np[k, 1], truth_np[k, 1])
        v = rmse_np(pred_np[k, 2], truth_np[k, 2])
        rows.append({
            "case": case,
            "method": method,
            "step": k,
            "time_hr": k * DT_MACRO / 3600.0,
            "eta_rmse": e,
            "u_rmse": u,
            "v_rmse": v,
            "total_rmse": total_rmse(e, u, v),
        })

# ------------------------------------------------------------
# Main loop
# ------------------------------------------------------------
all_rows = []
summary_rows = []

for ic_key in CASE_LIST:
    print("\n" + "=" * 80)
    print("[Case]", ic_key)
    print("=" * 80)

    truth_np = load_sequence(ic_key)
    truth = torch.tensor(truth_np, dtype=torch.float32, device=device)

    bg_np = make_background(truth_np[0], seed=42)
    mask_np = make_obs_mask(NY, NX, OBS_FRAC, seed=7)

    x_bg = torch.tensor(bg_np[None, ...], dtype=torch.float32, device=device)
    obs_mask = torch.tensor(mask_np[None, None, :, :], dtype=torch.float32, device=device)

    # Background rollout
    bg_roll = rollout_to_numpy(x_bg)

    # ML-3DVAR one-shot analysis
    x3_np = make_ml3dvar_input(bg_np, truth_np[0], mask_np)
    x3 = torch.tensor(x3_np[None, ...], dtype=torch.float32, device=device)

    with torch.no_grad():
        dx3 = ml3dvar(x3)
        x0_3d = x_bg + dx3

    ml3_roll = rollout_to_numpy(x0_3d)

    # ML-4DVAR analysis
    x0_4d, loss_hist = run_ml4dvar(x_bg, truth, obs_mask)
    ml4_roll = rollout_to_numpy(x0_4d)

    # Add time series rows
    add_rows(all_rows, ic_key, "background", bg_roll, truth_np)
    add_rows(all_rows, ic_key, "ml3dvar_unet", ml3_roll, truth_np)
    add_rows(all_rows, ic_key, "ml4dvar", ml4_roll, truth_np)

    # Final summary
    def final_metrics(pred):
        k = WINDOW_STEPS
        e = rmse_np(pred[k, 0], truth_np[k, 0])
        u = rmse_np(pred[k, 1], truth_np[k, 1])
        v = rmse_np(pred[k, 2], truth_np[k, 2])
        t = total_rmse(e, u, v)
        return e, u, v, t

    bg_e, bg_u, bg_v, bg_t = final_metrics(bg_roll)
    d3_e, d3_u, d3_v, d3_t = final_metrics(ml3_roll)
    d4_e, d4_u, d4_v, d4_t = final_metrics(ml4_roll)

    summary_rows.append({
        "case": ic_key,
        "obs_frac": OBS_FRAC,
        "window_steps": WINDOW_STEPS,

        "bg_eta": bg_e,
        "bg_u": bg_u,
        "bg_v": bg_v,
        "bg_total": bg_t,

        "ml3_eta": d3_e,
        "ml3_u": d3_u,
        "ml3_v": d3_v,
        "ml3_total": d3_t,

        "ml4_eta": d4_e,
        "ml4_u": d4_u,
        "ml4_v": d4_v,
        "ml4_total": d4_t,

        "ml3_eta_improve_pct": pct_improve(bg_e, d3_e),
        "ml3_total_improve_pct": pct_improve(bg_t, d3_t),
        "ml4_eta_improve_pct": pct_improve(bg_e, d4_e),
        "ml4_total_improve_pct": pct_improve(bg_t, d4_t),

        "ml4_final_loss": loss_hist[-1][1],
        "ml4_final_obs": loss_hist[-1][2],
    })

    print(
        f"[Final {ic_key}] "
        f"BG total={bg_t:.4f} | "
        f"ML3 total={d3_t:.4f} | "
        f"ML4 total={d4_t:.4f}"
    )

    # Save case plot: total RMSE over time
    case_rows = [r for r in all_rows if r["case"] == ic_key]
    plt.figure(figsize=(8, 5))
    for method in ["background", "ml3dvar_unet", "ml4dvar"]:
        rr = [r for r in case_rows if r["method"] == method]
        hrs = [r["time_hr"] for r in rr]
        vals = [r["total_rmse"] for r in rr]
        plt.plot(hrs, vals, marker="o", label=method)
    plt.xlabel("Time (hours)")
    plt.ylabel("Total RMSE")
    plt.title(f"ML-3DVAR vs ML-4DVAR | {ic_key}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{ic_key}_total_rmse_compare.png"), dpi=160)
    plt.close()

# ------------------------------------------------------------
# Save CSVs
# ------------------------------------------------------------
timeseries_csv = os.path.join(OUT_DIR, "ml3dvar_vs_ml4dvar_timeseries.csv")
with open(timeseries_csv, "w", newline="") as f:
    keys = list(all_rows[0].keys())
    w = csv.DictWriter(f, fieldnames=keys)
    w.writeheader()
    for r in all_rows:
        w.writerow(r)

summary_csv = os.path.join(OUT_DIR, "ml3dvar_vs_ml4dvar_summary.csv")
with open(summary_csv, "w", newline="") as f:
    keys = list(summary_rows[0].keys())
    w = csv.DictWriter(f, fieldnames=keys)
    w.writeheader()
    for r in summary_rows:
        w.writerow(r)

print("\n[Done]")
print("Timeseries:", timeseries_csv)
print("Summary:", summary_csv)
print("Plots:", OUT_DIR)

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[Load] emulator: /content/drive/MyDrive/AI_4DVAR2/training_runs/rescnn_b12_c96_lr1e4_t6_p4_keweak/final_model.pt
[Load] ML-3DVAR: /content/drive/MyDrive/AI_4DVAR2/ml3dvar_unet_1L_ckpt/best_model.pt
[Load] ML-3DVAR base_ch: 48

[Case] gauss_00
    [4DVAR 0000] loss=5.537133e-01 obs=5.537133e-01 bg=0.000000e+00 uv=0.000000e+00
    [4DVAR 0050] loss=2.272651e-01 obs=1.858555e-01 bg=1.494287e-01 uv=5.217063e-02
    [4DVAR 0100] loss=1.490692e-01 obs=1.025264e-01 bg=1.837031e-01 uv=5.523566e-02
    [4DVAR 0150] loss=1.234288e-01 obs=7.736775e-02 bg=1.904174e-01 uv=5.277363e-02
    [4DVAR 0200] loss=1.119029e-01 obs=6.728587e-02 bg=1.910265e-01 uv=4.969385e-02
    [4DVAR 0249] loss=1.059642e-01 obs=6.236085e-02 bg=1.891212e-01 uv=4.802408e-02
[Final gauss_00] BG total=4.8684 | ML3 total=4.0340 | ML4 total=4.2783

[Case] test_modon_00
    [4DVAR 0000] loss=9.077698e+00 obs=9.077698e+00 bg=0.000000e+00 uv=0.000000e+00
    [4D